# Taller Práctico de Criptografía Post-Cuántica
## ML-KEM y ML-DSA
### 6.° Encuentro de Ingenierías UPAEP 2026

En este notebook se demostrará el uso práctico de dos algoritmos estandarizados por NIST:

- **ML-KEM (FIPS 203):** mecanismo de encapsulación de claves, usado para establecer un secreto compartido y posteriormente cifrar/descifrar información.
- **ML-DSA (FIPS 204):** esquema de firma digital, usado para garantizar integridad y autenticidad.

## Objetivo del taller
Que el participante:
- ejecute ambos algoritmos,
- observe tamaños y tiempos,
- y comprenda la diferencia entre confidencialidad e integridad.

## 1. Descarga del material del taller

En esta sección se clona el repositorio de GitHub que contiene:

- los scripts de ML-KEM y ML-DSA,
- el archivo `message.txt`,
- y la documentación asociada.

### ¿Qué hacen los comandos?
- `rm -rf`: elimina una carpeta previa, si existía, para evitar conflictos.
- `git clone`: descarga el repositorio desde GitHub al entorno temporal de Colab.
- `%cd`: cambia el directorio de trabajo a la carpeta del proyecto.
- `ls`: lista los archivos disponibles para confirmar que el repositorio se descargó correctamente.

### Resultado esperado
Se debe observar en pantalla la lista de archivos del proyecto, incluyendo:
- `1_mlkem_cifrado.py`
- `2_mldsa_firma.py`
- `message.txt`

In [ ]:
%cd /content
!pwd
!ls

/content
/content
liboqs	sample_data  UPAEP-PQC-Workshop-MLKEM-MLDSA


In [ ]:
!rm -rf /content/UPAEP-PQC-Workshop-MLKEM-MLDSA
!git clone https://github.com/NullAstra404/UPAEP-PQC-Workshop-MLKEM-MLDSA.git
%cd /content/UPAEP-PQC-Workshop-MLKEM-MLDSA
!ls

Cloning into 'UPAEP-PQC-Workshop-MLKEM-MLDSA'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 64 (delta 28), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 32.08 KiB | 6.42 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/UPAEP-PQC-Workshop-MLKEM-MLDSA
1_mlkem_cifrado.py  docs     message.txt  requirements.txt
2_mldsa_firma.py    LICENSE  README.md


## 2. Instalación de dependencias del sistema

Antes de ejecutar los algoritmos post-cuánticos, es necesario instalar herramientas de compilación en el entorno de Colab.

### ¿Qué hacen los comandos?
- `apt-get update -qq`: actualiza la información de los paquetes del sistema.
- `apt-get install -y ...`: instala las herramientas necesarias:
  - `build-essential`: compilador y utilidades básicas de C/C++,
  - `cmake`: sistema de configuración de compilación,
  - `ninja-build`: herramienta rápida de construcción,
  - `git`: control de versiones, necesario para descargar bibliotecas.

### ¿Por qué es necesario?
Porque `liboqs` es una biblioteca nativa en C que debe compilarse antes de ser utilizada desde Python.

### Resultado esperado
El sistema debe reportar que los paquetes fueron instalados correctamente o que ya estaban disponibles.

In [ ]:
!apt-get update -qq
!apt-get install -y build-essential cmake ninja-build git

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
Suggested packages:
  gettext-base git-daemon-run | git-daemon-sysvinit git-doc git-email git-gui
  gitk gitweb git-cvs git-mediawiki git-svn
The following NEW packages will be installed:
  ninja-build
The following packages will be upgraded:
  git
1 upgraded, 1 newly installed, 0 to remove and 117 not upgraded.
Need to get 3,285 kB of archives.
After this operation, 358 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 git amd64 1:2.34.1-1ubuntu1.17 [3,174 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 ninja-bu

## 3. Compilación e instalación de liboqs

La biblioteca `liboqs` contiene las implementaciones nativas de ML-KEM, ML-DSA y otros algoritmos post-cuánticos.

### ¿Qué hacen los comandos?
- `rm -rf /content/liboqs`: elimina una copia previa para evitar inconsistencias.
- `git clone --depth=1 ...`: descarga el código fuente de `liboqs`.
- `cmake -S ... -B ...`: configura el proyecto para compilarlo.
- `cmake --build ...`: compila la biblioteca.
- `cmake --install ...`: instala la biblioteca compilada en `/usr/local`.
- `ldconfig`: actualiza la caché del enlazador dinámico para que el sistema pueda encontrar la librería.

### ¿Por qué es necesario?
El paquete de Python `liboqs-python` necesita encontrar la biblioteca compartida `liboqs.so`.  
Si esta biblioteca no existe o no es visible para el sistema, `import oqs` fallará.

### Resultado esperado
Al finalizar, `liboqs` debe quedar instalada en `/usr/local/lib/`.

In [ ]:
!rm -rf /content/liboqs
!git clone --depth=1 https://github.com/open-quantum-safe/liboqs /content/liboqs
!cmake -S /content/liboqs -B /content/liboqs/build -DBUILD_SHARED_LIBS=ON -DCMAKE_INSTALL_PREFIX=/usr/local
!cmake --build /content/liboqs/build --parallel 2
!cmake --install /content/liboqs/build
!ldconfig

Cloning into '/content/liboqs'...
remote: Enumerating objects: 2736, done.
remote: Counting objects: 100% (2736/2736), done.
remote: Compressing objects: 100% (1338/1338), done.
remote: Total 2736 (delta 1552), reused 1959 (delta 1367), pack-reused 0 (from 0)
Receiving objects: 100% (2736/2736), 2.63 MiB | 12.74 MiB/s, done.
Resolving deltas: 100% (1552/1552), done.
-- The C compiler identification is GNU 11.4.0
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Performing Test CC_SUPPORTS_WA_NOEXECSTACK
-- Performing Test CC_SUPPORTS_WA_NOEXECSTACK - Success
-- Performing Test LD_SUPPORTS_WL_Z_NOEXECSTACK
-- Performing Test LD_SUPPORTS_WL_Z_NOEXECSTACK - Success
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Foun

## 4. Verificación de la instalación de liboqs

Antes de usar Python, conviene verificar que la biblioteca realmente se instaló.

### ¿Qué hacen los comandos?
- `find /usr/local -name "liboqs*"`: busca archivos relacionados con `liboqs`.
- `ldconfig -p | grep oqs`: consulta si el sistema reconoce la biblioteca compartida instalada.

### Resultado esperado
Deben aparecer rutas como:
- `/usr/local/lib/liboqs.so`
- `/usr/local/lib/liboqs.so.0.15.0`

Esto confirma que la biblioteca nativa fue instalada correctamente.

In [ ]:
!find /usr/local -name "liboqs*" 2>/dev/null | sort
!ldconfig -p | grep oqs

/usr/local/lib/cmake/liboqs
/usr/local/lib/cmake/liboqs/liboqsConfig.cmake
/usr/local/lib/cmake/liboqs/liboqsConfigVersion.cmake
/usr/local/lib/cmake/liboqs/liboqsTargets.cmake
/usr/local/lib/cmake/liboqs/liboqsTargets-noconfig.cmake
/usr/local/lib/liboqs.so
/usr/local/lib/liboqs.so.0.15.0
/usr/local/lib/liboqs.so.9
/usr/local/lib/pkgconfig/liboqs.pc
	liboqs.so.9 (libc6,x86-64) => /usr/local/lib/liboqs.so.9
	liboqs.so (libc6,x86-64) => /usr/local/lib/liboqs.so


## 5. Prueba de carga directa de la biblioteca compartida

En esta sección se utiliza `ctypes` para comprobar que Python puede cargar directamente `liboqs.so`.

### ¿Qué hace este código?
- `ctypes.CDLL(...)`: intenta abrir la biblioteca compartida ubicada en `/usr/local/lib/liboqs.so`.

### ¿Por qué es útil?
Porque permite verificar si el problema está en la biblioteca nativa o en el wrapper de Python.

### Resultado esperado
Debe imprimirse:
`liboqs.so cargó correctamente`

In [ ]:
import ctypes
ctypes.CDLL("/usr/local/lib/liboqs.so")
print("liboqs.so cargó correctamente")

liboqs.so cargó correctamente


## 6. Configuración del entorno para Python

Ahora se configuran variables de entorno para que `liboqs-python` sepa dónde buscar la instalación de `liboqs`.

### ¿Qué hacen estas variables?
- `OQS_INSTALL_PATH`: indica la ruta base donde se instaló `liboqs`.
- `LD_LIBRARY_PATH`: agrega `/usr/local/lib` a la ruta de bibliotecas compartidas.

### ¿Por qué es necesario?
Porque el wrapper de Python necesita localizar `liboqs.so` al momento de ejecutar `import oqs`.

### Resultado esperado
Se deben imprimir las rutas configuradas.

In [ ]:
import os

os.environ["OQS_INSTALL_PATH"] = "/usr/local"
os.environ["LD_LIBRARY_PATH"] = "/usr/local/lib:/usr/local/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

print("OQS_INSTALL_PATH =", os.environ["OQS_INSTALL_PATH"])
print("LD_LIBRARY_PATH =", os.environ["LD_LIBRARY_PATH"])

OQS_INSTALL_PATH = /usr/local
LD_LIBRARY_PATH = /usr/local/lib:/usr/local/lib64:


## 7. Instalación del wrapper de Python

En esta sección se instala `liboqs-python`, que permite usar `liboqs` desde Python.

### ¿Qué hace el comando?
- `pip install -q liboqs-python`: instala el paquete de Python en modo silencioso.

### ¿Qué aporta este paquete?
Proporciona las clases y funciones para utilizar:
- ML-KEM
- ML-DSA
- otros algoritmos post-cuánticos
desde código Python.

### Resultado esperado
La instalación debe completarse sin errores.

In [ ]:
!pip install -q liboqs-python

## 8. Verificación del entorno post-cuántico

En esta sección se verifica que Python ya puede usar `oqs`, y se listan los algoritmos disponibles para el taller.

### ¿Qué hace este código?
- `import oqs`: carga el wrapper de Python.
- `get_enabled_kem_mechanisms()`: lista los mecanismos KEM disponibles.
- `get_enabled_sig_mechanisms()`: lista los algoritmos de firma disponibles.

### ¿Qué se espera observar?
- ML-KEM-512
- ML-KEM-768
- ML-KEM-1024
- ML-DSA-44
- ML-DSA-65
- ML-DSA-87

Esto confirma que el entorno quedó listo para ejecutar el taller.

In [ ]:
import oqs

print("OQS OK")

print("KEM:", [x for x in oqs.get_enabled_kem_mechanisms() if "ML-KEM" in x])

print("SIG:", [x for x in oqs.get_enabled_sig_mechanisms() if "ML-DSA" in x])

OQS OK
KEM: ['ML-KEM-512', 'ML-KEM-768', 'ML-KEM-1024']
SIG: ['ML-DSA-44', 'ML-DSA-65', 'ML-DSA-87']


## 9. Ejecución de ML-KEM

En este experimento se ejecuta el script `1_mlkem_cifrado.py`.

### ¿Qué hace este script?
1. Genera un par de llaves ML-KEM.
2. Realiza encapsulación para obtener:
   - un ciphertext KEM,
   - y un secreto compartido.
3. Realiza decapsulación para recuperar el mismo secreto compartido.
4. Deriva una clave simétrica.
5. Utiliza esa clave para cifrar y descifrar el archivo `message.txt`.

### ¿Qué propiedad demuestra?
**Confidencialidad**, porque el secreto compartido permite proteger el contenido del mensaje.

### ¿Qué resultado se espera?
- que el secreto compartido coincida,
- que el mensaje cifrado pueda descifrarse correctamente,
- y que se muestren tamaños y tiempos de ejecución.

In [ ]:
!python 1_mlkem_cifrado.py

/usr/local/lib/python3.12/dist-packages/oqs/__init__.py:1: UserWarning: liboqs version (major, minor) 0.15.0 differs from liboqs-python version 0.14.1
  from oqs.oqs import (

Parte 1 — ML-KEM: secreto compartido + cifrar/descifrar (demo)
Archivo de entrada: message.txt
Tamaño del mensaje (bytes): 246
Algoritmo KEM seleccionado: ML-KEM-768

Resultados
Secreto compartido generado correctamente: True
Mensaje cifrado y descifrado correctamente: True
¿Descifrado con clave incorrecta coincide con el original?: False

Tamaños (bytes)
 - Public Key:        1184
 - Secret Key:        2400
 - KEM Ciphertext:    1088
 - Shared Secret:     32
 - Ciphertext msg:    246

Tiempos (ms)
 - KeyGen:            1.767
 - Encapsulación:     0.169
 - Decapsulación:     0.065
 - Cifrado (XOR):     0.098
 - Descifrado correcto:0.078
 - Descifrado erróneo:0.076

Visualización del experimento
Mensaje original:
Taller de Criptografía Post-Cuántica
Universidad UPAEP – 2026

Este mensaje será utilizado para demost

## Interpretación de los resultados de ML-KEM

En este experimento se utilizó el algoritmo **ML-KEM (Module-Lattice Key Encapsulation Mechanism)** para generar un **secreto compartido** entre dos participantes.

El proceso se desarrolla en varias etapas:

### a. Generación de llaves
El sistema genera un **par de llaves**:

- **Public Key**
- **Secret Key**

La clave pública puede compartirse libremente, mientras que la clave secreta debe mantenerse privada.

---

### b. Encapsulación (Encapsulation)

Utilizando la **clave pública**, se genera:

- un **ciphertext del KEM**
- un **secreto compartido**

Este ciphertext se envía al receptor.

---

### c. Decapsulación (Decapsulation)

El receptor utiliza su **clave secreta** para recuperar el mismo **secreto compartido**.

Si todo funciona correctamente:
shared_secret_sender == shared_secret_receiver

Esto significa que ambas partes ahora poseen **la misma clave secreta**, sin haberla transmitido directamente.

---

### d. Derivación de clave simétrica

A partir del secreto compartido se genera una **clave simétrica** utilizando SHA-256.

clave = SHA256(shared_secret)


Esta clave se utiliza para cifrar y descifrar el mensaje.

---

### e. Cifrado del mensaje

El archivo `message.txt` se cifra utilizando un **keystream derivado de la clave**.

En la salida del programa se muestra:

- el **mensaje original**
- el **mensaje cifrado en hexadecimal**

El texto cifrado no es legible porque su contenido ha sido transformado criptográficamente.

---

### f. Descifrado con la clave correcta

Cuando se utiliza la **misma clave derivada del secreto compartido**, el sistema recupera el mensaje original.

Esto demuestra que:

Descifrado correcto = mensaje original


---

### g. Descifrado con una clave incorrecta

Para demostrar la propiedad de **confidencialidad**, el script intenta descifrar el mensaje usando una **clave incorrecta**.

El resultado es un texto sin sentido o con caracteres inválidos.

Esto ocurre porque **sin la clave correcta es computacionalmente inviable recuperar el mensaje original**.

---

## Propiedad criptográfica demostrada

Este experimento demuestra la propiedad de:

### **Confidencialidad**

Solo quien posee la **clave correcta derivada del secreto compartido** puede recuperar el mensaje original.

Cualquier intento de descifrado con una clave incorrecta produce un resultado inválido.

10. ## Firma digital con ML-DSA

En esta parte del taller se utilizará **ML-DSA (Module-Lattice Digital Signature Algorithm)** para demostrar cómo funcionan las firmas digitales post-cuánticas.

El proceso se divide en dos modos de ejecución:

### h. Generar la firma (`sign`)
En este paso se realizan tres operaciones:

- Se genera un **par de llaves** (pública y privada).
- Se firma el contenido del archivo `message.txt`.
- Se guardan en disco:
  - `mldsa_pk.bin` → llave pública  
  - `mldsa_sk.bin` → llave privada (solo para la demo)  
  - `message.sig.bin` → firma del mensaje  

### i. Verificar la firma (`verify`)
En este paso **no se genera una firma nueva**.  
Solo se verifica si el contenido actual de `message.txt` coincide con la firma guardada.

### Experimento

1. Ejecutar `sign` para crear la firma.
2. Ejecutar `verify` → la firma debe ser **VÁLIDA**.
3. Modificar `message.txt`.
4. Ejecutar `verify` nuevamente → la firma será **INVÁLIDA**.
5. Restaurar `message.txt`.
6. Ejecutar `verify` → la firma vuelve a ser **VÁLIDA**.

### Propiedad demostrada

Este experimento demuestra la propiedad de **integridad**:

> Si el contenido del mensaje cambia, la firma deja de ser válida.

In [ ]:
!python 2_mldsa_firma.py sign

/usr/local/lib/python3.12/dist-packages/oqs/__init__.py:1: UserWarning: liboqs version (major, minor) 0.15.0 differs from liboqs-python version 0.14.1
  from oqs.oqs import (

ML-DSA — Modo sign (generación de llaves y firma)
Archivo de entrada: message.txt
Tamaño del mensaje (bytes): 246
Algoritmo de firma seleccionado: ML-DSA-65

Resultados
Firma generada correctamente y guardada en disco.

Tamaños (bytes)
 - Public Key:  1952
 - Secret Key:  4032
 - Firma:       3309
 - Mensaje:     246

Tiempos (ms)
 - KeyGen:      2.713
 - Firma:       0.595

Archivos generados
 - mldsa_pk.bin   (llave pública)
 - mldsa_sk.bin   (llave privada - solo demo local)
 - message.sig.bin    (firma del mensaje)

Siguiente
Ahora ejecuta:
  python 2_mldsa_firma.py verify
Luego modifica message.txt y vuelve a ejecutar verify.
Finalmente restaura message.txt y vuelve a ejecutar verify.


## 11. Actividad guiada: modificar el mensaje

Ahora se modificará el contenido de `message.txt` para simular una alteración del mensaje original.

### ¿Qué se espera demostrar?
Que al cambiar el contenido, la firma ya no corresponde al mensaje y la verificación debe fallar.

In [ ]:
with open("message.txt", "w", encoding="utf-8") as f:
    f.write("""Taller de Criptografía Post-Cuántica
Universidad UPAEP – 2026

Este mensaje fue modificado durante la práctica.

Autor: Participante UPAEP
""")

print("message.txt fue modificado correctamente")

message.txt fue modificado correctamente


## 12. Verificación posterior a la modificación

Se vuelve a ejecutar el script de firma/verificación para observar el efecto del cambio en el mensaje.

### Resultado esperado
La verificación debe indicar que el contenido ya no coincide con la firma original.

In [ ]:
!python 2_mldsa_firma.py verify

/usr/local/lib/python3.12/dist-packages/oqs/__init__.py:1: UserWarning: liboqs version (major, minor) 0.15.0 differs from liboqs-python version 0.14.1
  from oqs.oqs import (

ML-DSA — Modo verify (verificación de firma existente)
Archivo de entrada: message.txt
Tamaño del mensaje (bytes): 144
Algoritmo de firma seleccionado: ML-DSA-65

Resultado de verificación
Verificación: INVÁLIDA ❌

Tamaños (bytes)
 - Public Key:  1952
 - Firma:       3309
 - Mensaje:     144

Tiempo (ms)
 - Verificación: 0.202

Interpretación
La firma NO corresponde al contenido actual de message.txt.
Esto indica que el mensaje fue modificado o que la firma/llave pública no corresponden.


## 13. Restauración del mensaje original

En esta sección se restablece el contenido original de `message.txt`.

### ¿Qué se busca demostrar?
Que al restaurar el contenido original y volver a ejecutar el proceso de firma/verificación, la firma vuelve a ser válida.

Esto permite observar claramente que la validez de la firma depende del contenido exacto del mensaje.

In [ ]:
with open("message.txt", "w", encoding="utf-8") as f:
    f.write("""Taller de Criptografía Post-Cuántica
Universidad UPAEP – 2026

Este mensaje será utilizado para demostrar:

1) Cifrado y descifrado utilizando ML-KEM.
2) Firma digital y verificación utilizando ML-DSA.

Autor: Israel Jaudy Pérez Bermúdez
""")

print("message.txt fue restaurado correctamente")

message.txt fue restaurado correctamente


## 14. Reejecución de la firma con el mensaje restaurado

Al restaurar el mensaje original y ejecutar nuevamente el script de ML-DSA, se espera observar que la firma vuelve a ser válida.

### Interpretación
Esto confirma que la firma digital depende del contenido exacto del mensaje:

- si el contenido cambia → la verificación falla
- si el contenido se restablece → la verificación vuelve a ser válida

In [ ]:
!python 2_mldsa_firma.py verify

/usr/local/lib/python3.12/dist-packages/oqs/__init__.py:1: UserWarning: liboqs version (major, minor) 0.15.0 differs from liboqs-python version 0.14.1
  from oqs.oqs import (

ML-DSA — Modo verify (verificación de firma existente)
Archivo de entrada: message.txt
Tamaño del mensaje (bytes): 246
Algoritmo de firma seleccionado: ML-DSA-65

Resultado de verificación
Verificación: VÁLIDA ✅

Tamaños (bytes)
 - Public Key:  1952
 - Firma:       3309
 - Mensaje:     246

Tiempo (ms)
 - Verificación: 0.199

Interpretación
La firma corresponde exactamente al contenido actual de message.txt.


## 15. Conclusión

En este taller se observó que:

- **ML-KEM** permite establecer un secreto compartido para posterior cifrado/descifrado.
- **ML-DSA** permite firmar y verificar digitalmente información.
- Ambos algoritmos son parte de la transición hacia criptografía post-cuántica y permiten discutir su impacto práctico en tiempos, tamaños y propiedades de seguridad.